# Hull Tactical Market Prediction - Improved Model

**Strategy:**
- Ensemble of LightGBM, XGBoost, and CatBoost
- Advanced feature engineering based on top solution analysis
- Target: market_forward_excess_returns

In [ ]:
import os
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

import kaggle_evaluation.default_inference_server

In [ ]:
# List input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

## Configuration

In [ ]:
DATA_PATH = Path('/kaggle/input/hull-tactical-market-prediction/')
MIN_SIGNAL = 0.0
MAX_SIGNAL = 2.0
SIGNAL_MULTIPLIER = 400.0

## Feature Engineering Functions

In [ ]:
def create_advanced_features(df: pl.DataFrame) -> pl.DataFrame:
    """Advanced feature engineering"""
    
    # Base features from top solution
    base_features = ["S2", "E2", "E3", "P9", "S1", "S5", "I2", "P8",
                     "P10", "P12", "P13"]
    
    # Additional promising features
    additional_features = ["M1", "M2", "M11", "V1", "V2", "E1", "E4", "E5",
                          "I1", "I7", "I9", "P1", "P2", "S3", "S4"]
    
    all_base = list(set(base_features + additional_features))
    
    # Create features
    df = df.with_columns([
        # Top solution features
        (pl.col("I2") - pl.col("I1")).alias("U1"),
        (pl.col("M11") / ((pl.col("I2") + pl.col("I9") + pl.col("I7")) / 3 + 1e-8)).alias("U2"),
        
        # Ratio features
        (pl.col("M1") / (pl.col("M2") + 1e-8)).alias("M_ratio"),
        (pl.col("P1") / (pl.col("P2") + 1e-8)).alias("P_ratio"),
        (pl.col("V1") / (pl.col("V2") + 1e-8)).alias("V_ratio"),
        
        # Group mean features
        ((pl.col("E1") + pl.col("E2") + pl.col("E3")) / 3).alias("E_mean"),
        ((pl.col("S1") + pl.col("S2") + pl.col("S5")) / 3).alias("S_mean"),
        ((pl.col("P8") + pl.col("P9") + pl.col("P10")) / 3).alias("P_mean"),
        
        # Interaction features
        (pl.col("S2") * pl.col("E2")).alias("SE_interact"),
        (pl.col("P9") * pl.col("M1")).alias("PM_interact"),
        (pl.col("I2") * pl.col("M11")).alias("IM_interact"),
    ])
    
    # Final feature list
    feature_cols = (all_base +
                   ["U1", "U2", "M_ratio", "P_ratio", "V_ratio",
                    "E_mean", "S_mean", "P_mean",
                    "SE_interact", "PM_interact", "IM_interact"])
    
    # Handle missing values using EWM
    for col in feature_cols:
        if col in df.columns:
            df = df.with_columns(
                pl.col(col).fill_null(pl.col(col).ewm_mean(com=0.5))
            )
    
    return df.select(["date_id", "target"] + feature_cols).drop_nulls()

In [ ]:
def convert_ret_to_signal(ret_arr):
    """Convert returns to signal"""
    return np.clip(ret_arr * SIGNAL_MULTIPLIER + 1, MIN_SIGNAL, MAX_SIGNAL)

## Load and Prepare Training Data

In [ ]:
# Load training data
train = (
    pl.read_csv(DATA_PATH / "train.csv")
    .rename({'market_forward_excess_returns': 'target'})
    .with_columns(pl.exclude('date_id').cast(pl.Float64, strict=False))
    .head(-10)
)

test_example = (
    pl.read_csv(DATA_PATH / "test.csv")
    .rename({'lagged_forward_returns': 'target'})
    .with_columns(pl.exclude('date_id').cast(pl.Float64, strict=False))
)

print(f"Train shape: {train.shape}")
print(f"Test shape: {test_example.shape}")

In [ ]:
# Create features
df = pl.concat([train, test_example], how="vertical")
df_processed = create_advanced_features(df)

train_processed = df_processed.filter(
    pl.col('date_id').is_in(train.get_column('date_id'))
)

FEATURES = [col for col in train_processed.columns if col not in ['date_id', 'target']]
print(f"Number of features: {len(FEATURES)}")

In [ ]:
# Prepare training data
X_train = train_processed.select(FEATURES).to_pandas().values
y_train = train_processed.get_column('target').to_numpy()

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"X_train shape: {X_train_scaled.shape}")
print(f"y_train shape: {y_train.shape}")

## Train Models

In [ ]:
# LightGBM
print("Training LightGBM...")
lgb_params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.03,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1
}

lgb_train = lgb.Dataset(X_train_scaled, y_train)
lgb_model = lgb.train(
    lgb_params,
    lgb_train,
    num_boost_round=500,
    valid_sets=[lgb_train],
    callbacks=[lgb.log_evaluation(0)]
)
print("LightGBM trained successfully!")

In [ ]:
# XGBoost
print("Training XGBoost...")
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmse',
    'max_depth': 5,
    'learning_rate': 0.03,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'tree_method': 'hist',
    'verbosity': 0
}

dtrain = xgb.DMatrix(X_train_scaled, label=y_train)
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=500
)
print("XGBoost trained successfully!")

In [ ]:
# CatBoost
print("Training CatBoost...")
cat_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.03,
    depth=5,
    l2_leaf_reg=3,
    random_seed=42,
    verbose=0
)
cat_model.fit(X_train_scaled, y_train, verbose=0)
print("CatBoost trained successfully!")

## Create Prediction Function

In [ ]:
def predict(test: pl.DataFrame) -> float:
    """Prediction function for inference server"""
    # Rename and process
    test = test.rename({'lagged_forward_returns':'target'})
    df_test = create_advanced_features(test)
    
    # Extract features
    X_test = df_test.select(FEATURES).to_pandas().values
    X_test_scaled = scaler.transform(X_test)
    
    # Ensemble prediction
    lgb_pred = lgb_model.predict(X_test_scaled, num_iteration=lgb_model.best_iteration)
    xgb_pred = xgb_model.predict(xgb.DMatrix(X_test_scaled))
    cat_pred = cat_model.predict(X_test_scaled)
    
    # Weighted average
    ensemble_pred = 0.5 * lgb_pred[0] + 0.25 * xgb_pred[0] + 0.25 * cat_pred[0]
    
    # Convert to signal
    signal = convert_ret_to_signal(ensemble_pred)
    
    return float(signal)

## Test Prediction Function

In [ ]:
# Test prediction on example data
test_result = predict(test_example.head(1))
print(f"Test prediction: {test_result}")
print(f"Signal range check: {MIN_SIGNAL} <= {test_result} <= {MAX_SIGNAL}")

## Start Inference Server

In [ ]:
inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))